# 01 — Data Check & Locked Splits (`MultiCamCows2024` + `NyalaData`)
Verifies files on disk, extracts camera IDs and basic image statistics (resolution, brightness, sharpness), and writes deterministic splits (seed 42) before any model runs.

In [ ]:
import json, os, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageStat

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Locate WildlifeReID-10k input folder on Kaggle
candidates = [
    Path("/kaggle/input/wildlifereid-10k"),
    Path("/kaggle/input/datasets/wildlifedatasets/wildlifereid-10k"),
]
DATA_ROOT = None
for c in candidates:
    if (c / "metadata.csv").exists():
        DATA_ROOT = c
        break
if DATA_ROOT is None:
    for p in Path("/kaggle/input").rglob("metadata.csv"):
        DATA_ROOT = p.parent
        break
assert DATA_ROOT is not None, f"metadata.csv not found under /kaggle/input: {list(Path('/kaggle/input').iterdir())}"
print("DATA_ROOT:", DATA_ROOT)

OUT = Path("/kaggle/working/out")
OUT.mkdir(parents=True, exist_ok=True)

meta = pd.read_csv(DATA_ROOT / "metadata.csv", low_memory=False)
print("Full WildlifeReID-10k shape:", meta.shape)

# 1. MultiCamCows2024
cows = meta[meta["dataset"] == "MultiCamCows2024"].copy().reset_index(drop=True)
cows["abs_path"] = cows["path"].apply(lambda p: str(DATA_ROOT / p))
cows["exists"] = cows["abs_path"].apply(os.path.exists)
assert cows["exists"].all(), f"Missing cow files: {(~cows['exists']).sum()}"
cows["camera"] = cows["path"].apply(lambda p: Path(p).name.split("_")[1])
cows["day"] = cows["date"].astype(str).str[:10]

# Partition 90 cows deterministically: 70 known, 10 val_unknown, 10 test_unknown
all_cow_ids = sorted(cows["identity"].unique())
assert len(all_cow_ids) == 90, f"Expected 90 cows, got {len(all_cow_ids)}"
rng = np.random.default_rng(SEED)
perm = rng.permutation(all_cow_ids)
val_unknown = set(perm[:10])
test_unknown = set(perm[10:20])
known_ids = set(perm[20:])

def cow_role(row):
    d = row["day"]
    ident = row["identity"]
    if d in {"2023-08-14", "2023-08-15", "2023-08-16"}:
        return "gallery" if ident in known_ids else "unused_early_unknown"
    elif d in {"2023-08-17", "2023-08-18"}:
        if ident in known_ids: return "val_known"
        if ident in val_unknown: return "val_unknown"
        return "unused_val_testunknown"
    elif d in {"2023-08-19", "2023-08-20"}:
        if ident in known_ids: return "test_known"
        if ident in test_unknown: return "test_unknown"
        return "unused_test_valunknown"
    return "other"

cows["role"] = cows.apply(cow_role, axis=1)
cows["id_group"] = cows["identity"].apply(
    lambda x: "known" if x in known_ids else ("val_unknown" if x in val_unknown else "test_unknown")
)

# Matched random split over known cows on days 14-18 (same query count as val_known)
pool_14_18 = cows[(cows["id_group"] == "known") & (cows["day"] <= "2023-08-18")].copy()
n_val_known = int((cows["role"] == "val_known").sum())
rand_idx = set(rng.choice(pool_14_18.index.to_numpy(), size=n_val_known, replace=False))
cows["random_split_role"] = ""
cows.loc[pool_14_18.index, "random_split_role"] = [
    "rand_query" if i in rand_idx else "rand_gallery" for i in pool_14_18.index
]

# Quick image integrity & brightness/resolution stats on a 300-image sample + verify all open cleanly
widths, heights, brights = [], [], []
corrupt = 0
for idx, p in enumerate(cows["abs_path"]):
    try:
        with Image.open(p) as im:
            w, h = im.size
            widths.append(w); heights.append(h)
            if idx % 10 == 0:
                brights.append(ImageStat.Stat(im.convert("L")).mean[0])
    except Exception as e:
        corrupt += 1
        print("Corrupt cow image:", p, e)
assert corrupt == 0, f"Found {corrupt} corrupt cow images"

# 2. NyalaData
nyala = meta[meta["dataset"] == "NyalaData"].copy().reset_index(drop=True)
nyala["abs_path"] = nyala["path"].apply(lambda p: str(DATA_ROOT / p))
nyala["exists"] = nyala["abs_path"].apply(os.path.exists)
assert nyala["exists"].all(), f"Missing nyala files: {(~nyala['exists']).sum()}"

# Save splits
cows.to_csv(OUT / "splits_cows.csv", index=False)
nyala.to_csv(OUT / "splits_nyala.csv", index=False)

summary = {
    "data_root": str(DATA_ROOT),
    "cows_total": int(len(cows)),
    "cows_unique_ids": int(cows["identity"].nunique()),
    "cows_role_counts": cows["role"].value_counts().to_dict(),
    "cows_cameras": cows["camera"].value_counts().to_dict(),
    "cows_days": cows["day"].value_counts().sort_index().to_dict(),
    "cows_random_split_counts": cows["random_split_role"].value_counts().to_dict(),
    "cows_median_wh": [int(np.median(widths)), int(np.median(heights))],
    "cows_mean_brightness_sample": round(float(np.mean(brights)), 2),
    "nyala_total": int(len(nyala)),
    "nyala_unique_ids": int(nyala["identity"].nunique()),
    "nyala_splits": nyala["split"].value_counts().to_dict(),
    "nyala_orientations": nyala["orientation"].value_counts().to_dict(),
}
(OUT / "data_check_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))